# rearrange-as-sequential-layer — worked example 3: unflatten with Rearrange in a decoder head

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rearrange-as-sequential-layer`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
from einops.layers.torch import Rearrange

## Concept

The inverse direction also works as a layer: a `Rearrange('b (c h w) -> b c h w', ...)` reshapes a flat latent into a spatial feature map at the start of a decoder, so a Linear can feed directly into a ConvTranspose stack without manual reshaping in forward.

## Worked solution

We build `decoder_head(latent_dim, channels, height, width)` as an `nn.Sequential`. A `Linear` expands the latent vector to `channels*height*width` features, then `Rearrange('b (c h w) -> b c h w', c=channels, h=height, w=width)` folds that flat vector back into a spatial map, and a final `ReLU` activates it. The named axis sizes tell einops how to split the flat dimension. We seed, build the head, pass a batch of latent vectors, and print the output shape, confirming the flat vector became `(B, channels, height, width)` with no reshape call in any forward method.

In [ ]:
import torch as t
from einops.layers.torch import Rearrange

t.manual_seed(2)

def decoder_head(latent_dim, channels, height, width):
    return t.nn.Sequential(
        t.nn.Linear(latent_dim, channels * height * width),
        Rearrange('b (c h w) -> b c h w', c=channels, h=height, w=width),
        t.nn.ReLU(),
    )

model = decoder_head(16, 4, 7, 7)
out = model(t.randn(3, 16))
print('output shape:', tuple(out.shape))  # (3, 4, 7, 7)